**Import Libraries**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize
from scipy.ndimage import binary_fill_holes
from scipy.interpolate import interp1d
from sympy import symbols, solve
from openpiv import tools, pyprocess, validation, filters, scaling
import pathlib
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import numpy as np
import cv2
import os
import alphashape
from shapely.geometry import Polygon, MultiPolygon
import math
mpl.rcParams['figure.dpi'] = 500

**Convert Video to Image Code**

video_path is the location of the video file. 

output_dir is the location of the image folder (should be empty initially).

startsec and endsec (in seconds) crops the video. 

In [ ]:
video_path = '/Users/braydennoh/Downloads/DISilica_Jan242025.MOV'
output_dir = '/Users/braydennoh/Documents/FlocTrackTest/images'
os.makedirs(output_dir, exist_ok=True)

startsec = 120
endsec = 421

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError("Error: Cannot open video file.")

fps = int(cap.get(cv2.CAP_PROP_FPS))
start_frame, end_frame = (t * fps for t in (startsec, endsec))

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
for frame_count in range(start_frame, end_frame):
    ret, frame = cap.read()
    if not ret:
        print(f"Frame {frame_count} could not be read. Exiting.")
        break
    cv2.imwrite(os.path.join(output_dir, f"frame_{frame_count}.png"), frame)

print(f"Frames saved in '{output_dir}'.")
cap.release()

In [ ]:
def compute_and_save_average_images(base_dir, start_frame, end_frame, step=1000, darken_factor=0.85):
    for i in range(start_frame, end_frame, step):
        images = [cv2.imread(os.path.join(base_dir, f"frame_{j}.png"), cv2.IMREAD_GRAYSCALE) for j in range(i, i + step)]
        images = [img.astype(np.float32) for img in images if img is not None]
        if images:
            avg_image = (sum(images) / len(images)) * darken_factor
            avg_image = np.clip(avg_image, 0, 255).astype(np.uint8)  # Ensure valid range
            avg_path = os.path.join(base_dir, f"average_image{i}.png")
            cv2.imwrite(avg_path, avg_image)
            print(f"Saved average image {avg_path}")
        else:
            print(f"No valid images for range {i}-{i + step}")
            
if __name__ == "__main__":
    compute_and_save_average_images(output_dir, 3500, 4500)

**Average Image Testing**

It should outline the particles you want to pick up after subtracting the average image. If it is capturing too much, decrease the darken_factor. If not enough, increase the darken_factor. 

In [ ]:
test_image_path = "/Users/braydennoh/Documents/FlocTrackTest/images/frame_5000.png"
average_image_path = "/Users/braydennoh/Documents/FlocTrackTest/images/average_image3500.png"

test_image = cv2.imread(test_image_path)
average_image = cv2.imread(average_image_path, cv2.IMREAD_GRAYSCALE)
test_gray = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)
if test_gray is None or average_image is None:
    raise FileNotFoundError("One or both images could not be loaded.")

clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8, 8))
test_enhanced = clahe.apply(test_gray)
average_enhanced = clahe.apply(average_image)
subtracted = test_enhanced - average_enhanced
subtracted_norm = cv2.normalize(subtracted, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
_, thresh = cv2.threshold(subtracted_norm, 50, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour_overlay = test_image.copy()
cv2.drawContours(contour_overlay, contours, -1, (0, 0, 255), 2)  # Red contours

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original Image")
ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(contour_overlay, cv2.COLOR_BGR2RGB))
ax[1].set_title("Contour Overlay")
ax[1].axis("off")
plt.show()

**Track Particles**

base_dir is the image subfolder. 

base_save_dir is the results subfolder (should be empty initially).

In [ ]:
import sys
sys.path.append("/Users/braydennoh/Documents/FlocTrackTest")


from concurrent.futures import ProcessPoolExecutor
import os
import cv2
import logging
from tqdm import tqdm
from ParticleProcess import process_frame_wrapper, load_average_image

# Set paths
base_dir = "/Users/braydennoh/Documents/FlocTrackTest/images"
base_save_dir = "/Users/braydennoh/Documents/FlocTrackTest/results"
average_image_path = "/Users/braydennoh/Documents/FlocTrackTest/images/average_image3500.png"
frame_range = range(3500, 12000)

# Load the single average image
average_image = load_average_image(average_image_path)
if average_image is None:
    raise FileNotFoundError(f"Average image not found: {average_image_path}")

# Prepare tasks
tasks = [(frame_num, average_image, base_dir, base_save_dir) for frame_num in frame_range]

logging.info(f"Processing {len(tasks)} frames.")

# Process frames in parallel
with ProcessPoolExecutor(max_workers=4) as executor:
    for _ in tqdm(executor.map(process_frame_wrapper, tasks), total=len(tasks)):
        pass  # Progress tracking

logging.info("Processing complete.")

**Link Tracked Particles**

In [ ]:
import os
import sys
sys.path.append("/Users/braydennoh/Documents/FlocTrackTest")
from ParticleTracking import track_particles
base_save_dir = "/Users/braydennoh/Documents/FlocTrackTest/results"
frame_nums =  range(3500, 4500) 
track_particles(base_save_dir, frame_nums)

**Connect Particles with minimal connection**

In [ ]:
from AnalyzeFrames import analyze_frames

base_save_dir = "/Users/braydennoh/Documents/FlocTrackTest/results"
frame_range = range(3500, 9000) 
min_connections = 10
averages_array, filtered_continuous_lines_data = analyze_frames(base_save_dir, frame_range, min_connections)
filtered_continuous_lines_data

**Add Velocity Column to Missing Columns**

In [ ]:
def check_and_add_velocity_column(base_dir, start_frame, end_frame):
    missing_velocity_frames = []  # To store frames where 'Velocity' was missing and added
    
    for frame_num in range(start_frame, end_frame + 1):
        # Construct the file path for the current frame
        frame_dir = os.path.join(base_dir, f'frame_{frame_num}')
        csv_path = os.path.join(frame_dir, 'contour_log.csv')
        
        # Check if the CSV file exists
        if not os.path.exists(csv_path):
            print(f"Frame {frame_num}: contour_log.csv not found.")
            continue
        
        # Load the CSV file
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Frame {frame_num}: Failed to read CSV file. Error: {e}")
            continue
        
        # Check if the 'Velocity' column exists
        if 'Velocity' in df.columns:
            print(f"Frame {frame_num}: 'Velocity' column exists.")
        else:
            print(f"Frame {frame_num}: 'Velocity' column is missing. Adding it now.")
            df['Velocity'] = None  # Add a blank 'Velocity' column
            try:
                df.to_csv(csv_path, index=False)  # Save the updated DataFrame back to the CSV
                missing_velocity_frames.append(frame_num)
            except Exception as e:
                print(f"Frame {frame_num}: Failed to save updated CSV. Error: {e}")
    
    # Summary of frames where 'Velocity' was missing and added
    if missing_velocity_frames:
        print("\nFrames where 'Velocity' column was missing and added:")
        print(missing_velocity_frames)
    else:
        print("\nAll frames have the 'Velocity' column.")

start_frame = 3500
end_frame = 4500

check_and_add_velocity_column(base_save_dir, start_frame, end_frame)

**Organize Tracking Data**

In [ ]:
pixels_per_micron = 0.45
fps = 30

def load_frame_data(frame_number):
    file_path = os.path.join(base_save_dir, f'frame_{frame_number}', 'contour_log.csv')
    if os.path.exists(file_path):
        return pd.read_csv(file_path)
    return pd.DataFrame()

def calculate_avg_diameter_velocity(path):
    particles = path.split(' -> ')
    concavediameters = []
    convexdiameters = []
    velocities = []
    positions_x = []
    positions_y = []
    perimeter_fractal_dimensions = []
    laplacian_list = []
    max_individual_diameter = 0 
    prediction = []
    for particle in particles:
        parts = particle.split(' in Frame ')
        particle_num = int(float(parts[0].split(' ')[1]))  
        frame_num = int(parts[1])
        
        frame_data = load_frame_data(frame_num)
        
        particle_data = frame_data[frame_data['ParticleNum'] == particle_num]
        
        if not particle_data.empty:
            concavediameter = particle_data['ConcaveHullDiameter'].values[0]
            concavediametermicron = concavediameter / pixels_per_micron
            concavediameters.append(concavediametermicron)
            
            convexdiameter = particle_data['KClusterDiameter'].values[0]
            convexdiametermicron = convexdiameter / pixels_per_micron
            convexdiameters.append(convexdiametermicron)
    
            curr_x = particle_data['X'].values[0]
            curr_y = particle_data['Y'].values[0]
            positions_x.append(curr_x)
            positions_y.append(curr_y)
            
            perimeter_fractal_dimensions.append(particle_data['PerimeterFractal'].values[0])
            laplacian_list.append(particle_data['LaplacianVariance'].values[0])
            avg_velocity = (particle_data['Velocity'].values[0] / pixels_per_micron) * fps
            if 'Velocity' in particle_data.columns:

                avg_velocity = (particle_data['Velocity'].values[0] / pixels_per_micron) * fps
                velocities.append(avg_velocity)
            else:
                velocities.append(np.nan) 
    
            individual_diameters_str = particle_data['IndividualDiameters'].values[0]
            if isinstance(individual_diameters_str, str):
                try:
                    individual_diameters = [float(d) for d in individual_diameters_str.split(',')]
                    max_individual_diameter = max(max_individual_diameter, max(individual_diameters))
                except ValueError:
                    pass 

    avg_concavediameter = np.nanmean(concavediameters) if concavediameters else 0
    avg_convexdiameter = np.nanmean(convexdiameters) if convexdiameters else 0
    avg_velocity = np.nanmean(velocities) if velocities else np.nan 
    avg_per_fractal_dimension = np.nanmean(perimeter_fractal_dimensions) if perimeter_fractal_dimensions else 0
    avg_laplacian = np.nanmean(laplacian_list) if laplacian_list else 0
    avg_prediction = round(np.nanmean(prediction)) if prediction else 0 
    return avg_concavediameter, avg_convexdiameter, avg_velocity, avg_per_fractal_dimension, avg_laplacian, max_individual_diameter

avg_concave_diameters = []
avg_convex_diameters = []
avg_velocities = []
avg_in_fractal_dimensions = []
avg_per_fractal_dimensions = []
avg_laplacian = []
max_individual_diameters = [] 

for path in filtered_continuous_lines_data:
    results = calculate_avg_diameter_velocity(path)
    avg_concave_diameters.append(results[0])
    avg_convex_diameters.append(results[1])
    avg_velocities.append(results[2])
    avg_per_fractal_dimensions.append(results[3])
    avg_laplacian.append(results[4])
    max_individual_diameters.append(results[5]) 

**Plotting Particle Diameter and Velocity**

In [ ]:
g = 9.81  # Acceleration due to gravity (m/s^2)
Rs = 1.65  # Submerged specific gravity of the primary particles
dp = 1e-6
b1 = 20  # Shape coefficient b1
b2 = 1.5  # Shape coefficient b2
nu = 1e-6  # Kinematic viscosity (m^2/s)
C1 = 18  # Coefficient for Stokes' law

all_diameters_corrected = [ad / 1000000 for ad in avg_concave_diameters]
all_velocities_corrected = [av / 1000000 for av in avg_velocities]

nf_values = [3.0, 2.5, 2.0, 1.5]

df = np.logspace(-6, -1, 100)

def settling_velocity(df, nf, g, Rs, dp, b1, b2, nu):
    numerator = g * Rs * df**(nf - 1)
    term1 = b1 * nu * dp**(nf - 3)
    term2 = b2 * np.sqrt(g * Rs * df**nf * dp**(nf - 3))
    return numerator / (term1 + term2)

def stokes_law(D):
    return (Rs * g * D**2) / (C1 * nu)

stokes_velocities = [stokes_law(d) for d in all_diameters_corrected]
velocity_ratios = np.array(all_velocities_corrected) / np.array(stokes_velocities)

plt.figure(figsize=(1.5,1.3))
ws_stokes = stokes_law(df)
plt.plot(df, ws_stokes, color='black', linestyle='--', label="Stokes' Law")
sc = plt.scatter(all_diameters_corrected, all_velocities_corrected,
                marker='o', s=3, alpha=0.9, edgecolor='none')
plt.xscale('log')
plt.yscale('log')
plt.ylabel('Velocity (m/s)')
plt.xlabel('Diameter (m)')
plt.ylim([0.00001, 0.1])
plt.xlim([0.000005, 0.001])
plt.show()